Add the following to your dataframe:

- Total sentences (use nltk sentence parser)
	Identify future-oriented sentences by looking for future oriented words
- FOG
- Total complex words (for the whole doc and for future tense)   >3 syllables
- Total Future Sentences
- every variable/column should have a future version (this doubles the number of columns, e.g. positivity and future_positivity)


Future oriented words:
Will
Shall
May
Might
Should
*Going to

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import nltk
nltk.download('punkt_tab')

import os
import re
import glob
import pandas as pd
from nltk.tokenize import sent_tokenize, word_tokenize

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# Syllable Counts

In [ ]:
CMUDICT_PATH = "/content/drive/MyDrive/Fall 2026 Text Mining/week 04/cmudict.txt"

cmudict = {}
with open(CMUDICT_PATH, encoding="latin-1") as f:
  for line in f:
    if line.startswith(";;;"):
      continue
    cols = line.split()
    if not cols:
      continue
    word = cols[0]
    if "(" in word:
      word = word[:word.index("(")]
    cmudict.setdefault(word.lower(), cols[1:])

print(len(cmudict))

125764


In [ ]:
def countVowelGroups(word):
  """Fallback syllable estimate for words that are not in CMUdict"""
  word = word.lower()
  groups = re.findall(r"[aeiouy]+", word)
  numSyllables = len(groups)
  if word.endswith("e") and not word.endswith(("le", "ee", "ye")) and numSyllables > 1:
    numSyllables -= 1
  return max(numSyllables, 1)


syllableCache = {}


def getSyllables(words):
  word = words.lower()
  if word in syllableCache:
    return syllableCache[word]

  phonemes = cmudict.get(word)
  if phonemes is None:
    syllables = countVowelGroups(word)
  else:
    numSyllables = sum(1 for p in phonemes if p[-1].isdigit())
    syllables = numSyllables

  syllableCache[word] = syllables
  return syllables


def isComplexWord(word):
  return getSyllables(word) > 2


print(getSyllables("Kevin"), isComplexWord("Kevin"))
print(getSyllables("readability"), isComplexWord("readability"))

2 False
5 True


# Future Sentences

In [ ]:
FUTURE_WORDS = {"will", "shall", "may", "might", "should"}


def isFuture(sentence, words=None):
  if words is None:
    words = [w.lower() for w in word_tokenize(sentence)]

  for word in words:
    if word in FUTURE_WORDS:
      return True

  # "going to" is two tokens; NLTK also splits "gonna" into "gon" + "na"
  for i in range(len(words) - 1):
    if words[i] == "going" and words[i + 1] == "to":
      return True
    if words[i] == "gon" and words[i + 1] == "na":
      return True

  return False


text = "NLTK is gonna be a great NLP toolkit. It makes processing text easy! Revenue will grow."
for s in sent_tokenize(text):
  print(isFuture(s), "|", s)

True | NLTK is gonna be a great NLP toolkit.
False | It makes processing text easy!
True | Revenue will grow.


# Sentiment word lists recap

In [ ]:
lm = pd.read_csv("/content/drive/MyDrive/Fall 2026 Text Mining/week 03/Lanqing Jia/Loughran-McDonald_MasterDictionary_1993-2025.csv")
lm["Word"] = lm["Word"].astype(str).str.lower()

positive  = set(lm.loc[lm["Positive"]  > 0, "Word"])
negative  = set(lm.loc[lm["Negative"]  > 0, "Word"])
litigious = set(lm.loc[lm["Litigious"] > 0, "Word"])

print(len(positive), len(negative), len(litigious))

354 2355 905


# Word Counter

In [ ]:
WORD_RE = re.compile(r"^[a-z][a-z']*$")

def analyzeDocument(text):
  stats = {
      "sentenceCount": 0, "future_sentenceCount": 0,
      "wordCount": 0,     "future_wordCount": 0,
      "complexWordCount": 0, "future_complexWordCount": 0,
      "syllableCount": 0, "future_syllableCount": 0,
      "PosWordCount": 0,  "future_PosWordCount": 0,
      "NegWordCount": 0,  "future_NegWordCount": 0,
      "LitWordCount": 0,  "future_LitWordCount": 0,
  }

  for sentence in sent_tokenize(text):
    tokens = [w.lower() for w in word_tokenize(sentence)]
    words = [w for w in tokens if WORD_RE.match(w)]
    if not words:
      continue

    future = isFuture(sentence, tokens)

    stats["sentenceCount"] += 1
    if future:
      stats["future_sentenceCount"] += 1

    for word in words:
      syl = getSyllables(word)
      complexWord = syl > 2

      stats["wordCount"] += 1
      stats["syllableCount"] += syl
      if complexWord:
        stats["complexWordCount"] += 1
      if word in positive:
        stats["PosWordCount"] += 1
      elif word in negative:
        stats["NegWordCount"] += 1
      elif word in litigious:
        stats["LitWordCount"] += 1

      if future:
        stats["future_wordCount"] += 1
        stats["future_syllableCount"] += syl
        if complexWord:
          stats["future_complexWordCount"] += 1
        if word in positive:
          stats["future_PosWordCount"] += 1
        elif word in negative:
          stats["future_NegWordCount"] += 1
        elif word in litigious:
          stats["future_LitWordCount"] += 1

  return stats

In [ ]:
def fogIndex(wordCount, sentenceCount, complexWordCount):
  """FOG = 0.4 * (words per sentence + percent complex words)"""
  if wordCount == 0 or sentenceCount == 0:
    return 0.0
  return 0.4 * ((wordCount / sentenceCount) + 100.0 * (complexWordCount / wordCount))


def rate(numerator, denominator):
  """Safe division: a document with no future sentences gets 0 instead of a crash."""
  if denominator == 0:
    return 0.0
  return numerator / denominator

# Build the df

In [ ]:
DOCS = "/content/drive/MyDrive/Fall2026TextMining_me/Lanqing10Ks"
files = sorted(glob.glob(os.path.join(DOCS, "*.txt")))
print(len(files), "files")

100 files


In [ ]:
rows = []

for i, path in enumerate(files, 1):
  text = open(path, "r", encoding="utf-8", errors="ignore").read()
  s = analyzeDocument(text)

  row = {
      "fileName": os.path.basename(path),

      # ---- whole document ----
      "wordCount":        s["wordCount"],
      "sentenceCount":    s["sentenceCount"],
      "complexWordCount": s["complexWordCount"],
      "PosWordCount":     s["PosWordCount"],
      "NegWordCount":     s["NegWordCount"],
      "LitWordCount":     s["LitWordCount"],
      "PosRate":          rate(s["PosWordCount"], s["wordCount"]),
      "NegRate":          rate(s["NegWordCount"], s["wordCount"]),
      "LitRate":          rate(s["LitWordCount"], s["wordCount"]),
      "complexRate":      rate(s["complexWordCount"], s["wordCount"]),
      "wordsPerSentence": rate(s["wordCount"], s["sentenceCount"]),
      "syllablesPerWord": rate(s["syllableCount"], s["wordCount"]),
      "FOG":              fogIndex(s["wordCount"], s["sentenceCount"], s["complexWordCount"]),

      # ---- future-oriented sentences only ----
      "future_wordCount":        s["future_wordCount"],
      "future_sentenceCount":    s["future_sentenceCount"],
      "future_complexWordCount": s["future_complexWordCount"],
      "future_PosWordCount":     s["future_PosWordCount"],
      "future_NegWordCount":     s["future_NegWordCount"],
      "future_LitWordCount":     s["future_LitWordCount"],
      "future_PosRate":          rate(s["future_PosWordCount"], s["future_wordCount"]),
      "future_NegRate":          rate(s["future_NegWordCount"], s["future_wordCount"]),
      "future_LitRate":          rate(s["future_LitWordCount"], s["future_wordCount"]),
      "future_complexRate":      rate(s["future_complexWordCount"], s["future_wordCount"]),
      "future_wordsPerSentence": rate(s["future_wordCount"], s["future_sentenceCount"]),
      "future_syllablesPerWord": rate(s["future_syllableCount"], s["future_wordCount"]),
      "future_FOG":              fogIndex(s["future_wordCount"], s["future_sentenceCount"], s["future_complexWordCount"]),

      # share of the document that is forward looking
      "futureSentenceRate": rate(s["future_sentenceCount"], s["sentenceCount"]),
      "futureWordRate":     rate(s["future_wordCount"], s["wordCount"]),
  }
  rows.append(row)
  print(i, row["fileName"], "sentences:", row["sentenceCount"], "future:", row["future_sentenceCount"])

df = pd.DataFrame(rows)
print(df.shape)

1 100_10-K_903419_2025-03-14.txt sentences: 16169 future: 751
2 10_10-K_1860782_2025-03-25.txt sentences: 12013 future: 1252
3 11_10-K_1860782_2025-03-17.txt sentences: 7 future: 3
4 12_10-K_1890702_2025-03-07.txt sentences: 617 future: 27
5 13_10-K_1948175_2025-03-10.txt sentences: 487 future: 24
6 14_10-K_933972_2025-03-28.txt sentences: 5 future: 1
7 15_10-K_910638_2025-03-27.txt sentences: 14414 future: 628
8 16_10-K_910638_2025-03-17.txt sentences: 13 future: 6
9 17_10-K_66740_2025-02-05.txt sentences: 17194 future: 904
10 18_10-K_1650648_2025-02-28.txt sentences: 12528 future: 1343
11 19_10-K_1783875_2025-03-31.txt sentences: 19 future: 7
12 1_10-K_1770787_2025-02-13.txt sentences: 62233 future: 918
13 20_10-K_1733443_2025-03-31.txt sentences: 7046 future: 592
14 21_10-K_1998387_2025-03-13.txt sentences: 10968 future: 3227
15 22_10-K_1946563_2025-03-27.txt sentences: 16222 future: 699
16 23_10-K_1785173_2025-02-27.txt sentences: 43557 future: 807
17 24_10-K_1555279_2025-03-07.txt

In [ ]:
df.head()

,fileName,wordCount,sentenceCount,complexWordCount,PosWordCount,NegWordCount,LitWordCount,PosRate,NegRate,LitRate,...,future_LitWordCount,future_PosRate,future_NegRate,future_LitRate,future_complexRate,future_wordsPerSentence,future_syllablesPerWord,future_FOG,futureSentenceRate,futureWordRate
0,100_10-K_903419_2025-03-14.txt,2546115,16169,227730,8420,6621,7620,0.003307,0.002600,0.002993,...,914,0.001851,0.002499,0.001716,0.047438,709.252996,1.394010,285.598733,0.046447,0.209201
1,10_10-K_1860782_2025-03-25.txt,696347,12013,134423,7030,5411,6272,0.010096,0.007771,0.009007,...,1380,0.009725,0.015250,0.012026,0.198116,91.654153,1.774974,44.586298,0.104220,0.164790
2,11_10-K_1860782_2025-03-17.txt,1649,7,156,0,17,9,0.000000,0.010309,0.005458,...,9,0.000000,0.010429,0.005521,0.095706,543.333333,1.534356,221.161554,0.428571,0.988478
3,12_10-K_1890702_2025-03-07.txt,794870,617,49543,50,532,560,0.000063,0.000669,0.000705,...,78,0.000029,0.000217,0.000376,0.054870,7678.777778,1.412860,3073.705905,0.043760,0.260831
4,13_10-K_1948175_2025-03-10.txt,609731,487,43194,56,381,477,0.000092,0.000625,0.000782,...,65,0.000048,0.000249,0.000449,0.070660,6035.333333,1.443941,2416.959744,0.049281,0.237560


In [ ]:
df.to_csv("/content/drive/MyDrive/Fall2026TextMining_me/readability_future.csv", index=False)